In [3]:
pip install control 

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pygame
import sys
import math
import numpy as np
import control

# ===== INITIALIZATION & SETUP =====
pygame.init()
pygame.font.init()
font = pygame.font.SysFont('Arial', 18)
pause_font = pygame.font.SysFont('Arial', 24, bold=True) 
large_font = pygame.font.SysFont('Arial', 60, bold=True) 

SIM_WIDTH = 1000
PLOT_WIDTH = 400
HEIGHT = 700
WIDTH = SIM_WIDTH + PLOT_WIDTH
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Pole Placement (100 DEG Fail Limit): Press SPACE")

BLACK = (0, 0, 0)
WHITE = (255, 255, 255)
RED = (255, 0, 0)
GREEN = (0, 255, 0)
BLUE = (0, 0, 255)
GREY = (100, 100, 100)
YELLOW = (255, 255, 0)

clock = pygame.time.Clock()
FPS = 60
dt = 1.0 / FPS

# ===== PHYSICS & SYSTEM PARAMETERS =====
GRAVITY = 9.8
CART_MASS = 1.0
POLE_MASS = 0.1
POLE_HALF_LENGTH = 0.5

FRICTION_CART = 0.1
FRICTION_POLE = 0.005

NOISE_STD = np.array([
    0.2,
    0.2,
    0.1,
    0.1
]).reshape(4, 1)

TOTAL_MASS = CART_MASS + POLE_MASS
POLE_MASS_LENGTH = POLE_MASS * POLE_HALF_LENGTH

PIXELS_PER_METER = 100
GROUND_Y = HEIGHT - 100
CART_WIDTH_PX = 40
CART_HEIGHT_PX = 20
POLE_LENGTH_PX = POLE_HALF_LENGTH * 2 * PIXELS_PER_METER
WHEEL_RADIUS_PX = 8
WHEEL_RADIUS_M = WHEEL_RADIUS_PX / PIXELS_PER_METER
origin_x_px = SIM_WIDTH // 2 

# ===== STATE-SPACE MODEL & CONTROLLER =====
M, m, l, g = CART_MASS, POLE_MASS, POLE_HALF_LENGTH, GRAVITY
denom = (m + 4 * M)
A = np.zeros((4, 4))
B = np.zeros((4, 1))
A[0, 1] = 1.0
A[1, 2] = (-3 * m * g) / denom
A[2, 3] = 1.0
A[3, 2] = (3 * g * (M + m)) / (l * denom)
B[1, 0] = 4 / denom
B[3, 0] = -3 / (l * denom)

print("--- Ideal Linearized State-Space Model ---")
print("A Matrix:\n", np.round(A, 4))
print("\nB Matrix:\n", np.round(B, 4))

desired_poles = [-1.0, -1.5, -3.0, -4.0]

K = control.place(A, B, desired_poles)

print("\n--- Pole Placement Controller ---")
print("Desired Poles:", desired_poles)
print("Original K (Gain Matrix):\n", K) 

K[0, 0] = 0.0

print("\n--- Modified Controller (K[0,0] = 0) ---") 
print("Modified K (Gain Matrix):\n", K)
print("\n*** SIMULATION RUNNING (HIGH NOISE, NO POS CONTROL) ***\n")
print("--------------------------------------\n")

# ===== INITIAL STATE & DATA STORAGE =====
cart_x_m = 0.0
cart_v_ms = 0.0
pole_angle_rad = 0.0
pole_v_rads = 0.0
force = 0.0

angle_history = []
force_history = []

paused = True 
failure_state = False
FAILURE_ANGLE_RAD = math.radians(100.0)

# ===== MAIN LOOP =====
running = True
while running:
    # ===== INPUT HANDLING =====
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        
        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_SPACE:
                paused = not paused
                failure_state = False
                if not paused:
                    mouse_x, mouse_y = pygame.mouse.get_pos()
                    mouse_x_clamped = np.clip(mouse_x, 0, SIM_WIDTH)
                    cart_x_m = (mouse_x_clamped - origin_x_px) / PIXELS_PER_METER
                    pole_angle_rad = np.interp(mouse_y, [0, HEIGHT], [-FAILURE_ANGLE_RAD, FAILURE_ANGLE_RAD])
                    cart_v_ms = 0.0
                    pole_v_rads = 0.0
                    angle_history.clear()
                    force_history.clear()

    # ===== SIMULATION LOGIC (PHYSICS + CONTROL) =====
    if paused:
        if not failure_state:
            mouse_x, mouse_y = pygame.mouse.get_pos()
            mouse_x_clamped = np.clip(mouse_x, 0, SIM_WIDTH)
            cart_x_m = (mouse_x_clamped - origin_x_px) / PIXELS_PER_METER
            
            pole_angle_rad = np.interp(mouse_y, [0, HEIGHT], [-FAILURE_ANGLE_RAD, FAILURE_ANGLE_RAD])
            
            force = 0.0
            cart_v_ms = 0.0
            pole_v_rads = 0.0

            angle_history.clear()
            force_history.clear()

    else:
        true_state_x = np.array([
            [cart_x_m], [cart_v_ms], [pole_angle_rad], [pole_v_rads]
        ])
        
        noise = (np.random.randn(4, 1) * NOISE_STD)
        measured_state_x = true_state_x + noise
        
        force_ideal = -K @ measured_state_x
        force = force_ideal[0, 0]
        
        force_friction_cart = -FRICTION_CART * cart_v_ms
        pole_damping_acc = -FRICTION_POLE * pole_v_rads

        sin_theta = math.sin(pole_angle_rad)
        cos_theta = math.cos(pole_angle_rad)
        
        net_force = force + force_friction_cart 
        
        temp_term = (net_force + POLE_MASS_LENGTH * pole_v_rads**2 * sin_theta) / TOTAL_MASS
        theta_acc = (GRAVITY * sin_theta - cos_theta * temp_term) / (
                    POLE_HALF_LENGTH * (4.0/3.0 - (POLE_MASS * cos_theta**2) / TOTAL_MASS)
        )
        theta_acc += pole_damping_acc
        x_acc = temp_term - (POLE_MASS_LENGTH * theta_acc * cos_theta) / TOTAL_MASS
        
        cart_v_ms += x_acc * dt
        cart_x_m += cart_v_ms * dt
        pole_v_rads += theta_acc * dt
        pole_angle_rad += pole_v_rads * dt

        if abs(pole_angle_rad) > FAILURE_ANGLE_RAD:
            paused = True
            failure_state = True
        
        if not failure_state:
            angle_history.append(pole_angle_rad)
            force_history.append(force)
            
            def trim_history(history, max_len):
                if len(history) > max_len:
                    history.pop(0)
            
            trim_history(angle_history, PLOT_WIDTH)
            trim_history(force_history, PLOT_WIDTH)

    # ===== RENDERING: SIMULATION VISUALIZATION (LEFT SIDE) =====
    screen.fill(BLACK)
    
    pygame.draw.line(screen, WHITE, (0, GROUND_Y), (SIM_WIDTH, GROUND_Y), 2)
    pygame.draw.line(screen, GREY, (origin_x_px, 0), (origin_x_px, HEIGHT), 1)

    cart_center_x_px = origin_x_px + (cart_x_m * PIXELS_PER_METER)
    cart_top_y_px = GROUND_Y - CART_HEIGHT_PX - (WHEEL_RADIUS_PX * 2)
    pivot_x_px = cart_center_x_px
    pivot_y_px = cart_top_y_px
    pole_end_x_px = pivot_x_px + POLE_LENGTH_PX * math.sin(pole_angle_rad)
    pole_end_y_px = pivot_y_px - POLE_LENGTH_PX * math.cos(pole_angle_rad)
    
    cart_rect = pygame.Rect(
        cart_center_x_px - CART_WIDTH_PX // 2,
        cart_top_y_px, CART_WIDTH_PX, CART_HEIGHT_PX
    )
    pygame.draw.rect(screen, GREEN, cart_rect)
    pygame.draw.line(screen, RED, 
                        (pivot_x_px, pivot_y_px), 
                        (pole_end_x_px, pole_end_y_px), 8)
    pygame.draw.circle(screen, RED, (int(pole_end_x_px), int(pole_end_y_px)), 10)
    pygame.draw.circle(screen, WHITE, (int(pivot_x_px), int(pivot_y_px)), 5)
    
    wheel_angle_rad = cart_x_m / PIXELS_PER_METER / WHEEL_RADIUS_M
    wheel1_cx_px = int(cart_center_x_px - CART_WIDTH_PX // 4)
    wheel2_cx_px = int(cart_center_x_px + CART_WIDTH_PX // 4)
    wheel_cy_px = int(GROUND_Y - WHEEL_RADIUS_PX)
    pygame.draw.circle(screen, BLUE, (wheel1_cx_px, wheel_cy_px), WHEEL_RADIUS_PX)
    pygame.draw.circle(screen, BLUE, (wheel2_cx_px, wheel_cy_px), WHEEL_RADIUS_PX)
    
    spoke1_end_x = wheel1_cx_px + WHEEL_RADIUS_PX * math.cos(wheel_angle_rad)
    spoke1_end_y = wheel_cy_px + WHEEL_RADIUS_PX * math.sin(wheel_angle_rad)
    pygame.draw.line(screen, WHITE, (wheel1_cx_px, wheel_cy_px), (spoke1_end_x, spoke1_end_y), 2)
    spoke2_end_x = wheel2_cx_px + WHEEL_RADIUS_PX * math.cos(wheel_angle_rad)
    spoke2_end_y = wheel_cy_px + WHEEL_RADIUS_PX * math.sin(wheel_angle_rad)
    pygame.draw.line(screen, WHITE, (wheel2_cx_px, wheel_cy_px), (spoke2_end_x, spoke2_end_y), 2)

    # ===== RENDERING: PLOTS AREA (RIGHT SIDE) =====
    plot_x_start = SIM_WIDTH
    plot_h_mid = HEIGHT // 2 
    
    pygame.draw.line(screen, WHITE, (plot_x_start, 0), (plot_x_start, HEIGHT), 2)
    pygame.draw.line(screen, WHITE, (plot_x_start, plot_h_mid), (WIDTH, plot_h_mid), 2)

    # ===== PLOT 1: POLE ANGLE (TOP GRAPH) =====
    angle_plot_center_y = plot_h_mid // 2 
    angle_scale = 50 
    angle_label = font.render('Pole Angle (rad)', True, WHITE)
    screen.blit(angle_label, (plot_x_start + 10, 10))
    pygame.draw.line(screen, GREY, (plot_x_start, angle_plot_center_y), (WIDTH, angle_plot_center_y), 1)
    
    angle_points = []
    for i, angle in enumerate(angle_history):
        angle_points.append((plot_x_start + i, angle_plot_center_y - (angle * angle_scale)))
    if len(angle_points) > 1:
        pygame.draw.lines(screen, RED, False, angle_points, 2)

    # ===== PLOT 2: APPLIED FORCE (BOTTOM GRAPH) =====
    force_plot_center_y = plot_h_mid + (plot_h_mid // 2)
    force_scale = 2 
    force_label = font.render('Applied Force (N)', True, WHITE)
    screen.blit(force_label, (plot_x_start + 10, plot_h_mid + 10)) 
    pygame.draw.line(screen, GREY, (plot_x_start, force_plot_center_y), (WIDTH, force_plot_center_y), 1)
    
    force_points = []
    for i, f in enumerate(force_history):
        force_points.append((plot_x_start + i, force_plot_center_y - (f * force_scale)))
    if len(force_points) > 1:
        pygame.draw.lines(screen, YELLOW, False, force_points, 2)

    # ===== UI OVERLAY (PAUSE / FAILURE MESSAGES) =====
    if paused:
        if failure_state:
            msg = "Pendulum ko itna neeche se matt choro CUTIE"
            bg_color = (150, 0, 0, 220)
            
            text_surface = large_font.render(msg, True, WHITE)
            text_rect = text_surface.get_rect(center=(WIDTH // 2, HEIGHT // 2)) 
            bg_rect = text_rect.inflate(40, 20)
            
            bg_surf = pygame.Surface(bg_rect.size, pygame.SRCALPHA)
            bg_surf.fill(bg_color)
            
            screen.blit(bg_surf, bg_rect)
            screen.blit(text_surface, text_rect)
            
        else:
            msg = "MOVE MOUSE TO SET STATE. PRESS SPACE TO START."
            bg_color = (0, 0, 0, 180)
            
            text_surface = pause_font.render(msg, True, WHITE)
            text_rect = text_surface.get_rect(center=(SIM_WIDTH // 2, 40)) 
            bg_rect = text_rect.inflate(20, 10)
            
            bg_surf = pygame.Surface(bg_rect.size, pygame.SRCALPHA)
            bg_surf.fill(bg_color)
            
            screen.blit(bg_surf, bg_rect)
            screen.blit(text_surface, text_rect)

    pygame.display.flip()
    clock.tick(FPS)

pygame.font.quit()
pygame.quit()
sys.exit()

--- Ideal Linearized State-Space Model ---
A Matrix:
 [[ 0.      1.      0.      0.    ]
 [ 0.      0.     -0.7171  0.    ]
 [ 0.      0.      0.      1.    ]
 [ 0.      0.     15.7756  0.    ]]

B Matrix:
 [[ 0.    ]
 [ 0.9756]
 [ 0.    ]
 [-1.4634]]

--- Pole Placement Controller ---
Desired Poles: [-1.0, -1.5, -3.0, -4.0]
Original K (Gain Matrix):
 [[ -1.25510204  -2.82397959 -32.80006803  -8.37431973]]

--- Modified Controller (K[0,0] = 0) ---
Modified K (Gain Matrix):
 [[  0.          -2.82397959 -32.80006803  -8.37431973]]

*** SIMULATION RUNNING (HIGH NOISE, NO POS CONTROL) ***

--------------------------------------



SystemExit: 

C:\Users\fa071\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
